In [1]:
import ipdb # <- трасировка и точки останова

In [2]:
from header import __root__
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to load GAPI credentials


In [3]:
import importlib
import asyncio
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, Dict, Any, List

from pydoll.browser.chrome import Chrome
from pydoll.constants import By
from pydoll.browser.page import Page

from src.llm.gemini import GoogleGenerativeAi # Unused, but kept
from src.endpoints.prestashop.product_fields import ProductFields
from src.endpoints.prestashop.product_async import PrestaProductAsync
from src.endpoints.prestashop.product import PrestaProduct

from src.webdriver.pydoll_driverless import execute_locator

from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory
from src.utils.jjson import j_loads, j_loads_ns, j_dumps # j_dumps unused
from src.utils.image import get_image_bytes, get_raw_image_data 
from src.utils.printer import pprint as print
from src.logger.logger import logger

In [4]:
class Config:
    """Класс конфигурации скрипта."""
    ENDPOINT: Path = __root__ / 'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    SCENARIOS_DIR: Path = __root__ / 'SANDBOX' / 'davidka' / 'scenarios'
    # config: SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json') #  general config.
    scenarios_files: List[str] = get_filenames_from_directory(SCENARIOS_DIR) # SANDBOX/davidka/scenarios/*.json
    PRESTA_API_KEY: str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_API_DOMAIN: str = gs.credentials.prestashop.store_davidka_net.api_domain
    presta_product_async: PrestaProductAsync = PrestaProductAsync(api_key=PRESTA_API_KEY, api_domain=PRESTA_API_DOMAIN)
    presta_product: PrestaProduct = PrestaProduct(api_key=PRESTA_API_KEY, api_domain=PRESTA_API_DOMAIN)
    browser: Chrome = None
    page: Page = None

In [5]:
browser = Config.browser
if not browser:
    browser = Chrome()  
    await browser.start()
    
page = Config.page    
if not page:
    page = await browser.get_page()
        

2025-06-13 02:54:26,571 - INFO - EventsHandler initialized
2025-06-13 02:54:26,571 - INFO - ConnectionHandler initialized.
2025-06-13 02:54:26,878 - INFO - Connecting to ws://localhost:9311/devtools/browser/443690ed-932e-447d-adcf-572319da101c
2025-06-13 02:54:28,932 - INFO - EventsHandler initialized
2025-06-13 02:54:28,933 - INFO - ConnectionHandler initialized.


##### execute_locator
Не забыть перенести в код

In [22]:
async def fetch_product_fields(page: Page, actual_fields:Optional[list] = None) -> ProductFields:
    """Grab product fields."""

    ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list' / 'aliexpress' 

    actual_fields:list = ['id_supplier',                                                              
                         'name',
                         'price',
                         'reference',
                         'description',
                         'description_short',
                         'default_image_url',
                         'local_image_path',]

    locator:SimpleNamespace = j_loads_ns(ENDPOINT / 'locators' / 'product.json')

    async def save_local_image(f) -> bool:
        """Fetch and save an image locally.

        Функция получает `URL` картинки или байты изображения, сохраняет изображение в формате `PNG` в директории `tmp` 
        и устанавливает путь к сохранённой картинке в поле `local_image_path`.
        """
        try:
            # Получаем результат из локатора как `bytes` или `str`(url)
            image_url:str = f.default_image_url
            img_path:Path = Path(gs.path.tmp / f'{f.id_supplier}_{f.reference}.png')
            await save_image_from_url_async(image_url, img_path)
            return img_path
        except Exception as ex:
            logger.error(f'Ошибка сохранения изображения в поле `local_image_path`', ex)
            ...
            return None


    f:ProductFields = ProductFields()

    f.id_supplier = locator.id_supplier.attribute
    f.name = await execute_locator(page, locator.name)
    f.reference = page.current_url.split("/item/")[1].split(".html")[0]
    f.price = await execute_locator(page, locator.price)

    if 'description' in actual_fields:
        f.description = await execute_locator(page, locator.description)
    if 'description_short' in actual_fields:
        f.description_short = await execute_locator(page, locator.description_short)
    if 'default_image_url' in actual_fields:
        f.default_image_url = await execute_locator(page, locator.default_image_url)
    if 'local_image_path' in actual_fields:
        f.local_image_path = await save_local_image(f)

    return f

async def grab_product_page( page: Page, product_url: str, actual_fields:Optional[list] = None) -> ProductFields:
    """
    Загружает страницу товара по URL и возвращает структуру данных ProductFields.
    Поддерживаются входные URL формата:
        //he.aliexpress.com/item/
        https://he.aliexpress.com/item/
        he.aliexpress.com/item/
    """

    if product_url.startswith('//'):
        url = f'https:{product_url}'
    elif product_url.startswith('http://') or product_url.startswith('https://'):
        url = product_url
    else:
        url = f'https://{product_url.lstrip("/")}'

    await page.go_to(url)
    return await fetch_product_fields(product_url, page, actual_fields or Config.actual_fields)

async def get_product_urls_from_category_page(category_url:str, locator:SimpleNamespace, page: Page) -> List[str]:
    """Get product URLs from the current page.
   Отдельная функция для каждого поставщика, так как локаторы могут отличаться.
    """
    await page.go_to(category_url)
    product_urls = await execute_locator(page, locator)
    return product_urls


In [6]:
# ПРАВИЛЬНО:
async def __save_to_prestashop_async(f:ProductFields):
    """"""
    async with Config.presta_product_async as presta_product_async:
        # Теперь presta_product_api.client инициализирован
        result = await presta_product_async.add_new_product_async(f)


async def save_to_prestashop_async(f:ProductFields):
    """"""
    p = Config.presta_product
    print(f.to_dict())
    ipdb.set_trace()
    result = await p.add_new_product_async(f)
    

#### process supplier

In [ ]:
async def ___get_product_urls_from_category_page(category_url:str, locator:SimpleNamespace, page: Page) -> List[str]:
    """Get product URLs from the current page.
   Отдельная функция для каждого поставщика, так как локаторы могут отличаться.
    """
    print(locator)
    await page.go_to(category_url)
    product_urls = await execute_locator(page, locator)
    return product_urls

In [7]:
def get_graber(supplier_alias) -> Any:
    graber_module_path:str  = f"src.suppliers.suppliers_list.{supplier_alias}.graber_via_pydoll"
    try:
        graber = importlib.import_module(graber_module_path)
        return graber
    except Exception as ex:
        logger.error(f"Failed to import module `graber` '{supplier_prefix}'", ex)
        return None    

In [8]:
async def process_supplier(supplier_prefix:str, page: 'Page', product_url:Optional[str] = None ) -> bool:
    """Название файла JSON соответствуют `supplier_prefix`, а  названия папок в системе - `supplier_alias` """
    ...
    
    try:
        supplier_alias:str = supplier_prefix.replace('.','_').replace('-','_')
        supplier_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_alias 
        product_locators:SimpleNamespace = j_loads_ns(supplier_path / 'locators' / 'product.json')
        category_locators:SimpleNamespace = j_loads_ns(supplier_path / 'locators' / 'category.json')
        actual_fields:list = ['id_supplier',                                                              
                     'name',
                     'price',
                     'reference',
                     'description',
                     'description_short',
                     'default_image_url',
                     'local_image_path']
        # --- dev ---
        scenarios_list: list = j_loads(Config.SCENARIOS_DIR  / f'{supplier_prefix}.json') # <- ЧИТАЮ ИЗ ПАПКИ САНДБОХ
    except Exception as ex:
        
        logger.error(f'Непредвиденная ошибка', ex)
        return False


    graber = get_graber(supplier_alias)
    
    if not graber:
        return False

    if product_url: # <- обработка одной ссылки
        f:ProductFields = await graber.grab_product_page(page, product_url, actual_fields)
        return await save_to_prestashop_async(f)
        
    for scenario in scenarios_list:
        products_urls_in_category:list = await graber.get_product_urls_from_category_page(scenario['url'], category_locators.product_links, page)

        if not products_urls_in_category:
            logger.debug(f'Вероятно, пустая категория ')
            print(scenario)
            continue # <- мб пустаая категория
            ...

        for product_url in products_urls_in_category:
            f:ProductFields = await graber.grab_product_page(page, product_url, actual_fields)
            await save_to_prestashop_async(f)
            
        return True


In [ ]:
supplier_prefix:str= 'aliexpress'
# await process_supplier(supplier_prefix, page)

### Точечные проверки

In [16]:
ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list' / 'aliexpress'
product_locator = j_loads_ns(ENDPOINT / 'locators' / 'product.json')
category_locator = j_loads_ns(ENDPOINT / 'locators' / 'category.json')


In [ ]:

#product_url:str = 'https://he.aliexpress.com/item/1005004869497167.html?algo_pvid=5bcf218a-5626-40fd-99ed-9ede265fa590&algo_exp_id=5bcf218a-5626-40fd-99ed-9ede265fa590-0&pdp_ext_f=%7B%22order%22%3A%22172%22%2C%22eval%22%3A%221%22%7D&pdp_npi=4%40dis%21ILS%21812.41%21568.70%21%21%21224.18%21156.93%21%40212e508f17497439898337727ea5e6%2112000030824283710%21sea%21IL%210%21ABX&curPageLogUid=G8tFBGu1NjyI&utparam-url=scene%3Asearch%7Cquery_from%3A#nav-specification'
#category_url = 'https://www.aliexpress.com/w/wholesale-industrial-servo-motors.html?spm=a2g0o.productlist.search.0'
#res = await get_product_urls_from_category_page(category_url, category_locator.product_links, page)


In [ ]:
#res = await execute_locator(page, category_locator.product_links)

In [23]:
actual_fields:list = [
    'id_supplier',                                                              
     'name',
     'price',
     'reference',
     'description',
     'description_short',
     'default_image_url',
     'local_image_path',
    ]
product_fields = await fetch_product_fields(page, actual_fields)

AttributeError: 'coroutine' object has no attribute 'split'

In [ ]:
graber_module_path:str  = f"src.suppliers.suppliers_list.{supplier_prefix}.graber_via_pydoll"
graber = importlib.import_module(graber_module_path)
await graber.grab_product_page(page, product_url)

In [ ]:
await page.go_to(product_url)

In [ ]:
actual_fields:list = ['id_supplier',                                                              
                     'name',
                     'price',
                     'reference',
                     'description',
                     'description_short',
                     'default_image_url']

f:ProductFields = ProductFields()

f.id_supplier = locator.id_supplier.attribute
f.name = await execute_locator(page, locator.name)
f.reference = product_url.split("/item/")[1].split(".html")[0]

f.price = await execute_locator(page, locator.price)


if 'description' in actual_fields:
    f.description = await execute_locator(page, locator.description)
if 'description_short' in actual_fields:
    f.description_short = await execute_locator(page, locator.description_short)
if 'default_image_url' in actual_fields:
    f.default_image_url = await execute_locator(page, locator.default_image_url)

In [ ]:
print(f.to_dict())

In [ ]:
await save_to_prestashop(f)